# safetensors

- safetensors 是由 Hugging Face 团队开发的一种专门用于安全、高效存储和加载张量（Tensors，即模型权重）的文件格式。它的诞生正是为了解决传统 PyTorch .bin / .pth 格式（基于 Python pickle 模块）存在的安全漏洞和性能瓶颈。
- 简单来说，.safetensors 是一个“更安全、更快、且跨框架兼容的纯模型权重存储文件”。

## 🛡️ 为什么 .safetensors 成为 Hugging Face 的新标准？
1. 极高的安全性（核心优势）

- 传统的 .bin 格式在加载时会使用 pickle 进行反序列化，这个过程可能会执行文件里隐藏的任意恶意代码（例如窃取系统密钥或植入木马）。
- 而 .safetensors 在设计上彻底杜绝了代码执行的可能性。<font color='red'>它只允许保存和读取“纯张量数据”和 JSON 格式的元数据，没有任何可执行代码的生存空间</font>。这意味着即使你从不可信的第三方下载模型，加载 .safetensors 文件也是绝对安全的。

2. 极致的加载速度（零拷贝与内存映射）

- 加载 .safetensors 采用了“零拷贝（zero-copy）”和内存映射（mmap）技术。
   - <font color='red'>传统方式：加载 .bin 时，Python 需要将文件复制到内存 -> 进行反序列化 -> 转换回张量格式 -> 移动到 GPU，存在大量不必要的复制过程。</font>
   - Safetensors：<font color='green'>操作系统直接将文件映射到内存地址，数据可以从磁盘立即被使用，无需经过 Python 解释器繁琐的反序列化过程。在加载像 GPT-3 这样拥有数千亿参数的大模型时，.safetensors 的加载速度比 pickle 快数倍，且内存占用大幅降低。</font>
    
3. 跨框架兼容与延迟加载
   - 它的底层结构非常简洁，不依赖于特定的深度学习框架。不仅 PyTorch 能用，TensorFlow、JAX (Flax)、PaddlePaddle 等主流框架都能直接读取同一份权重文件。
   - 此外，<font color='red'>它支持延迟加载（Lazy Loading）。如果你只想从一个 100GB 的大模型文件中读取某几层的权重，.safetensors 可以通过读取头部信息，精准定位并只加载你需要的部分，而无需将整个庞大的文件读入内存。</font>

## 📂 .safetensors 的底层文件结构
- safetensors 的文件结构非常直观，主要由以下三部分组成：
1. 8字节的头部大小：一个无符号小端序（little-endian）64位整数，记录了接下来元数据头部的大小。
2. JSON 元数据头部：包含每个张量的名称、数据类型（dtype）、形状（shape）以及该张量数据在文件中的起始和结束偏移量（data_offsets）。
3. 二进制数据缓冲区：紧跟在头部之后，是一块连续的二进制数据区，密集地填充着所有张量的实际数值。

<font color='red'>得益于这种结构，程序只需读取开头的 JSON 头部，就能完全掌握模型的结构并快速定位任意一个权重，而无需扫描整个文件。</font>

## 💻 如何在代码中使用 .safetensors？
- 你可以通过 pip install safetensors 安装官方库。以下是 PyTorch 框架下的基本使用示例：

### 保存模型权重：

In [ ]:
import torch
from safetensors.torch import save_file

# 假设这是你的模型权重字典
tensors = {
    "weight1": torch.ones((4, 4), dtype=torch.int32),
    "weight2": torch.zeros((8, 8), dtype=torch.int32)
}
# 保存为 .safetensors 文件
save_file(tensors, "model.safetensors")

### 加载模型权重（全量加载）

In [ ]:
from safetensors.torch import load_file

# 一次性加载整个文件到指定设备（如 cpu 或 cuda:0）
loaded_weights = load_file("model.safetensors", device="cpu")
print(loaded_weights["weight1"])

### 延迟/部分加载（适合超大模型）：

In [ ]:
from safetensors import safe_open

# 使用 safe_open 可以按需读取，无需一次性载入内存
with safe_open("model.safetensors", framework="pt", device="cpu") as f:
    # 获取某个张量的切片或部分数据
    tensor_slice = f.get_slice("weight2") # 只获取weight2
    shape = tensor_slice.get_shape()
    # 例如只读取前 4 行的数据
    partial_tensor = tensor_slice[:4, :] 

## 💡 总结
- 目前，Hugging Face Hub 上发布的主流大模型（如 Llama 3、Qwen、Stable Diffusion、Flux 等）都已默认提供或优先推荐使用 .safetensors 版本。在模型推理和生产部署阶段（如使用 vLLM、SGLang 等框架），强烈建议优先使用 .safetensors 格式，它不仅能有效防范供应链安全攻击，还能大幅缩短服务冷启动时间并节省宝贵的服务器内存。

# bin(PyTorch)

- 在深度学习领域，尤其是 Hugging Face 和 PyTorch 生态中，.bin 文件（通常命名为 pytorch_model.bin）是一种非常经典的模型权重存储格式。
- 简单来说，.bin 文件是 PyTorch 使用其原生的序列化机制保存下来的一个“纯二进制数据包”，里面装满了模型训练学到的所有参数（权重和偏置）。

## 📦 .bin 格式的核心原理：<font color='red'>基于 Pickle 的序列化</font>
- .bin 文件的底层完全依赖于 Python 自带的 pickle 模块。
- 当你使用 PyTorch 的 torch.save() 函数保存模型时，pickle 会把你的模型权重（通常是一个 Python 字典，即 state_dict）以及相关的元数据，直接“腌制（pickle）”成一段连续的二进制流，并写入到 .bin 文件中。

- 包含内容：<font color='red'>.bin 文件通常只包含模型的权重参数（张量数据），而不包含模型的结构代码。因此，在加载 .bin 文件时，你必须先在代码中定义好完全一致的模型架构，才能把权重“灌”进去。</font>
- 加载方式：<font color='red'>使用 torch.load("pytorch_model.bin") 时，pickle 会进行“反序列化（unpickling）”，将二进制流重新还原成 Python 的字典对象（OrderedDict），其中的键（key）是层名称（如 model.layers.0.weight），值（value）是对应的张量（Tensor）。</font>

## ⚠️ .bin 格式的致命缺陷：安全隐患
- 这是 .bin 格式目前面临的最大争议点。由于 pickle 的设计初衷是序列化任意 Python 对象，<font color='red'>它在反序列化时会无条件执行文件中包含的 Python 代码。</font>
- 这意味着，.bin 文件存在极大的任意代码执行（RCE）风险。如果你从不可信的第三方（如陌生的 Hugging Face 仓库或网盘）下载了一个 .bin 模型文件，攻击者可以在文件中植入恶意代码（例如窃取你服务器上的 SSH 密钥、环境变量或植入木马）。当你运行 torch.load() 加载该文件时，这些恶意代码就会在你的机器上自动执行。

## 🐢 性能瓶颈：加载慢且内存占用高
- 除了安全问题，.bin 格式在处理当今动辄数百 GB 的大语言模型（LLM）时，性能表现也显得力不从心：
- 加载速度慢：<font color='red'>Python 的 pickle 在反序列化时需要经过解释器逐层解析，无法像 .safetensors 那样利用操作系统的内存映射（mmap）技术实现“零拷贝”加载。</font>
- 内存峰值高：<font color='red'>加载过程中，系统往往需要先将整个文件读入内存，重建出完整的 Python 对象后，再转移到 GPU，</font>这会导致瞬时的内存（CPU RAM）占用极高，容易引发内存溢出（OOM）。

| 维度 | PyTorch .bin 格式 |
| :--- | :--- |
| 底层机制 | 依赖 Python `pickle` 序列化协议 |
| 包含内容 | <font color='red'>仅模型权重参数（state_dict），不含模型结构</font> |
| 安全性 | 极低（存在任意代码执行漏洞，需警惕来源不明的文件） |
| 加载效率 | 较慢（需 Python 解释器反序列化，无法零拷贝） |
| 适用场景 | <font color='red'>传统的 PyTorch 训练检查点、旧版模型存档 </font>|

## 💡 总结与建议
- .bin 格式作为 PyTorch 生态的“老将”，在过去几年中承载了无数模型的发布与分享。它的优势在于与 PyTorch 框架的原生兼容性极好，且在模型训练阶段（保存包含优化器状态、学习率调度器的完整检查点时）依然非常灵活。
- 但在模型推理和生产部署阶段，由于严重的安全漏洞和加载性能瓶颈，它正在被迅速淘汰。目前，Hugging Face 以及 vLLM、SGLang 等主流推理框架都强烈建议优先使用 .safetensors 格式。如果你手头有 .bin 格式的模型，建议在受信任的环境中将其转换为 .safetensors 后再进行分发或部署，以确保安全与高效。

# 零拷贝（Zero-copy）

- 零拷贝（Zero-copy）是一种计算机操作系统优化技术，<font color='red'>它的核心目的是减少或消除数据在不同内存区域之间不必要的复制。</font>
- <font color='red'>在计算机系统中，数据经常需要在磁盘、内核缓冲区、用户程序（应用层）和网络接口（网卡）之间来回搬运</font>。传统的处理方式往往伴随着大量的数据拷贝，这不仅消耗宝贵的 CPU 周期，还会占用大量的内存带宽。
- 零拷贝的本质，就是<font color='red'>把“复制数据内容”尽可能转化为“传递数据引用（比如内存地址、指针或描述符）”</font>。简单来说，就是让<font color='red'>数据只在一处存在，所有需要它的人共享同一个引用，而不是每个人都去复印一份。</font>

## 📦 传统的数据拷贝：为什么它很慢？

- 为了理解零拷贝的价值，我们先看看传统的文件传输（比如从磁盘读取一个文件并通过网络发送出去）是怎么工作的。这通常需要经历 4 次数据拷贝 和 4 次上下文切换（在用户态和内核态之间反复横跳）：
1. DMA 拷贝：DMA（直接内存访问）引擎把数据从磁盘读取到内核缓冲区。
2. CPU 拷贝：CPU 把数据从内核缓冲区复制到用户缓冲区（你的应用程序内存中）。
3. CPU 拷贝：CPU 再把数据从用户缓冲区复制回内核 Socket 缓冲区。
4. DMA 拷贝：DMA 引擎最后把数据从 Socket 缓冲区复制到网卡（网络接口），发送出去。

你会发现，第 2 步和第 3 步是极其浪费的。<font color='red'>应用程序往往只是数据的“搬运工”，并没有对数据进行实质性的加工，却被迫让 CPU 亲力亲为地把数据搬进搬出。</font>

## ⚡ 零拷贝是如何实现的？

零拷贝技术通过<font color='red'>绕过用户空间（应用程序内存），让数据直接在内核空间完成流转</font>，从而大幅降低开销。以下是几种主流的零拷贝实现方式：
1. mmap + write：减少一次 CPU 拷贝
- mmap（内存映射）将内核缓冲区直接映射到用户进程的虚拟地址空间。
- 原理：应用程序不再需要调用 read() 把数据复制到用户空间，而是直接通过映射的内存地址访问内核缓冲区的数据。
- 效果：消除了“内核缓冲区到用户缓冲区”的那次 CPU 拷贝，但数据发送到网卡时，仍然需要从内核缓冲区拷贝到 Socket 缓冲区。

2. sendfile：真正的内核态直接传输
- sendfile 是一个专门用于在两个文件描述符之间传输数据的系统调用（例如直接从文件描述符传到网络 Socket）。
- 原理：数据全程不经过用户空间。内核直接把数据从磁盘的内核缓冲区，传递到对应网卡的 Socket 缓冲区。
- 效果：不仅消除了两次上下文切换，还减少了一次 CPU 拷贝。相比传统方式，它只需要 3 次拷贝（其中 2 次是 DMA，1 次是 CPU）。

3. sendfile + 带聚集操作的 DMA (SG-DMA)：极致的零 CPU 拷贝
- 这是现代 Linux 内核（2.4版本以后）配合支持 SG-DMA 的网卡实现的终极形态。
- 原理：当数据需要从内核缓冲区发送到网卡时，CPU 连最后一次拷贝都不做了。内核只把数据的“内存地址和长度信息（描述符）”传给 Socket 缓冲区。支持聚集操作的 DMA 引擎会直接根据这些地址信息，从内核缓冲区中读取数据并发送到网卡。
- 效果：全程只有 2 次 DMA 拷贝（磁盘->内核缓冲区，内核缓冲区->网卡），CPU 完全不需要参与任何数据的搬运工作，实现了真正的“零 CPU 拷贝”。

| 传输方式 | 上下文切换次数 | CPU 拷贝次数 | 核心特点 |
| :--- | :--- | :--- | :--- |
| 传统 `read` + `write` | 4 次 | 2 次 | 数据在用户态和内核态反复折腾，开销最大 |
| `mmap` + `write` | 4 次 | 1 次 | 共享内核内存，减少一次拷贝，但仍有 CPU 参与 |
| `sendfile` | 2 次 | 1 次 | 绕过用户态，数据在内核直接流转 |
| `sendfile` + SG-DMA | 2 次 | 0 次 | 真正的零 CPU 拷贝，硬件 DMA 全程接管 |

## 💡 零拷贝带来的好处
- 大幅降低 CPU 负载：CPU 从繁重的数据搬运工作中解放出来，可以去处理更复杂的业务逻辑（如加密、压缩或计算）。
- 减少内存带宽占用：减少了不必要的数据复制，意味着内存总线不再被冗余数据堵塞。
- 降低延迟，提升吞吐量：更少的上下文切换和数据拷贝，直接带来了更快的响应速度和更高的并发处理能力。
- <font color='red'>这也是为什么像 Nginx、Kafka 这样的高性能服务器软件，以及现代深度学习推理框架（如 vLLM、SGLang 在底层数据传输时）都会深度依赖零拷贝技术来榨干硬件性能的原因。</font>